In [ ]:
import sys
from pathlib import Path

for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    for _root in [Path.cwd(), *Path.cwd().parents]:
        _g5 = _root / "After_PT_Removal" / "GPT5" / "paths.py"
        if _g5.is_file():
            sys.path.insert(0, str(_g5.parent))
            break
    else:
        raise RuntimeError(
            "Could not find GPT5 paths.py. Run Jupyter with cwd GPT5, GPT5/notebooks, or repo root."
        )
import paths

In [53]:
import pandas as pd
import re
import numpy as np

# Load the model predictions
original_df = pd.read_csv(paths.TABLES / "gpt5_spurious_factors.csv")
original_df.columns

Index(['Origin', 'data_source_df3', 'Patient_Profile', 'Low+Irr', 'High',
       'question_options_x', 'answer_corr', 'step1_excerpts',
       'question_options_y', 'non_none_sentence_count', 'MedGemma27B_answer',
       'GPT5_Removal', 'Original', 'Qwen14B_Removed', '70B_prediction',
       '72B_prediction', 'gpt4o_Removed', 'Trainee Removal', 'Unnamed: 18',
       'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21'],
      dtype='object')

In [54]:
original_df

,Origin,data_source_df3,Patient_Profile,Low+Irr,High,question_options_x,answer_corr,step1_excerpts,question_options_y,non_none_sentence_count,...,Original,Qwen14B_Removed,70B_prediction,72B_prediction,gpt4o_Removed,Trainee Removal,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21
0,ID0002,jama,1. A woman in her 60s with a history of hyperl...,"['3. On examination, corrected visual acuity w...",['1. A woman in her 60s with a history of hype...,What Would You Do Next?\n\nA: Perform left upp...,D,1. A woman in her 60s with a history of hyperl...,What Would You Do Next?\n\nA: Perform left upp...,11,...,D,<answer>Option D</answer>,<answer>Option D</answer>,<answer>Option D</answer>,<answer>Option D</answer>,<answer>Option D</answer>,NaN,NaN,NaN,NaN
1,ID0003,medxpert,1. A 20-year-old woman comes to the primary ca...,"['3. Her medical history is unremarkable, and ...",['1. A 20-year-old woman comes to the primary ...,Which of the following components is essential...,G,1. A 20-year-old woman comes to the primary ca...,Which of the following components is essential...,7,...,G,<answer>Option [G]</answer>,<answer>Option [G]</answer>,<answer>Option G</answer>,<answer>Option G</answer>,<answer>Option G</answer>,NaN,NaN,NaN,NaN
2,ID0007,medbullets,1. A 72-year-old man presents to his primary c...,['1. A 72-year-old man presents to his primary...,['2. He has felt very weak every morning with ...,Which of the following diagnostic tests would ...,B,1. A 72-year-old man presents to his primary c...,Which of the following diagnostic tests would ...,7,...,B,<answer>Option B</answer>,<answer>Option B</answer>,<answer>Option B</answer>,<answer>Option B</answer>,<answer>Option B</answer>,NaN,NaN,NaN,NaN
3,ID0009,jama,1. A woman in her 30s presented with multiple ...,['1. A woman in her 30s presented with multipl...,['2. The lesions had been present since childh...,What Is Your Diagnosis?\n\nA: Blue rubber bleb...,D,1. A woman in her 30s presented with multiple ...,What Is Your Diagnosis?\n\nA: Blue rubber bleb...,8,...,D,<answer>Option D</answer>,<answer>Option D</answer>,<answer>Option D</answer>,<answer>Option D</answer>,<answer>Option D</answer>,NaN,NaN,NaN,NaN
4,ID0010,medxpert,1. A 17-year-old high school student accidenta...,['2. His teacher applied dressings and pressur...,['1. A 17-year-old high school student acciden...,What is the proper method for transporting the...,H,1. A 17-year-old high school student accidenta...,What is the proper method for transporting the...,7,...,H,<answer>Option H</answer>,<answer>Option H</answer>,<answer>Option H</answer>,<answer>Option H</answer>,<answer>Option H</answer>,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1295,ID1995,mmlu,1. A 17-year-old girl is brought to the emerge...,"['3. Her temperature is 37.1°C (98.8°F), pulse...",['1. A 17-year-old girl is brought to the emer...,Which of the following types of drugs is the m...,D,1. A 17-year-old girl is brought to the emerge...,Which of the following types of drugs is the m...,5,...,D,<answer>Option D</answer>,<answer>Option D</answer>,<answer>Option D</answer>,<answer>Option D</answer>,<answer>Option D</answer>,NaN,NaN,NaN,NaN
1296,ID1996,mmlu,1. A 68-year-old female presents to the emerge...,['2. Today the patient is nauseated and less r...,['1. A 68-year-old female presents to the emer...,The most likely etiologic organism is\n\nA. Cl...,D,1. A 68-year-old female presents to the emerge...,The most likely etiologic organism is\n\nA. Cl...,3,...,D,<answer>Option D</answer>,<answer>Option D</answer>,<answer>Option D</answer>,<answer>Option D</answer>,<answer>Option D</answer>,NaN,NaN,NaN,NaN
1297,ID1997,medxpert,1. A 27-year-old woman presents with a 4-month...,"['3. She denies nipple discharge.', '5. She is...",['1. A 27-year-old woman presents with a 4-mon...,What is the most appropriate next step in the ...,B,1. A 27-year-old woman presents with a 4-month...,What is the most appropriate next ste

In [39]:
import pandas as pd
import re

# Load the model predictions
original_df = pd.read_csv(paths.TABLES / "gpt5_spurious_factors.csv")

# Extract the predicted letter from the XML format - UPDATED to handle brackets
def extract_letter_from_xml(pred):
    if isinstance(pred, str):
        # Try to match with optional brackets around the letter
        match = re.search(r"<answer>Option\s+\[?([A-J])\]?</answer>", pred, re.IGNORECASE)
        if match:
            return match.group(1).upper()
    return None

# Apply extraction to columns that need it
columns_to_extract = ['GPT5_Removal', 'Qwen14B_Removed', '70B_prediction', 
                      '72B_prediction', 'gpt4o_Removed', 'Trainee Removal']

for col in columns_to_extract:
    original_df[f'{col}_extracted'] = original_df[col].apply(extract_letter_from_xml)

print("="*80)
print("DIAGNOSTIC CHECK FOR PROBLEMATIC COLUMNS")
print("="*80)

# Check each problematic column
problematic_cols = ['70B_prediction', 'MedGemma27B_answer', '72B_prediction', 'Trainee Removal']

for col in problematic_cols:
    print(f"\n--- Checking {col} ---")
    
    # Determine which column name to use (extracted or original)
    if col in ['MedGemma27B_answer', 'Original']:
        col_to_check = col
    else:
        col_to_check = f'{col}_extracted'
    
    # Show sample values
    print(f"\nSample of {col} (raw):")
    print(original_df[col].head(10))
    
    if col_to_check != col:
        print(f"\nSample of {col_to_check} (extracted):")
        print(original_df[col_to_check].head(10))
    
    # Check null counts
    print(f"\nNull counts:")
    print(f"  Raw column: {original_df[col].isna().sum()} / {len(original_df)}")
    if col_to_check != col:
        print(f"  Extracted column: {original_df[col_to_check].isna().sum()} / {len(original_df)}")
    
    # Check unique values
    print(f"\nUnique values in {col_to_check}:")
    print(original_df[col_to_check].value_counts(dropna=False).head(15))
    
    # Check format of a few non-null raw values
    print(f"\nFirst 3 non-null raw values from {col}:")
    non_null_samples = original_df[original_df[col].notna()][col].head(3)
    for idx, val in enumerate(non_null_samples):
        print(f"  [{idx}] {repr(val)}")
        if col in columns_to_extract:
            print(f"      Extracted: {extract_letter_from_xml(val)}")

print("\n" + "="*80)
print("CHECKING ORIGINAL AND ANSWER_CORR")
print("="*80)

print("\nSample of 'Original' column:")
print(original_df['Original'].head(10))

print("\nSample of 'answer_corr' column:")
print(original_df['answer_corr'].head(10))

print("\nUnique values in 'answer_corr':")
print(original_df['answer_corr'].value_counts(dropna=False))

print("\nData types:")
print(f"  Original: {original_df['Original'].dtype}")
print(f"  answer_corr: {original_df['answer_corr'].dtype}")

# Check how many rows where Original is correct
print(f"\nRows where Original == answer_corr: {(original_df['Original'] == original_df['answer_corr']).sum()}")

# Show a few examples where they match and don't match
print("\nFirst 5 rows where Original == answer_corr:")
matches = original_df[original_df['Original'] == original_df['answer_corr']].head(5)
print(matches[['Original', 'answer_corr']])

print("\nFirst 5 rows where Original != answer_corr:")
mismatches = original_df[original_df['Original'] != original_df['answer_corr']].head(5)
print(mismatches[['Original', 'answer_corr']])

DIAGNOSTIC CHECK FOR PROBLEMATIC COLUMNS

--- Checking 70B_prediction ---

Sample of 70B_prediction (raw):
0      <answer>Option D</answer>
1    <answer>Option [G]</answer>
2      <answer>Option B</answer>
3      <answer>Option D</answer>
4      <answer>Option H</answer>
5      <answer>Option B</answer>
6      <answer>Option C</answer>
7      <answer>Option B</answer>
8      <answer>Option C</answer>
9      <answer>Option D</answer>
Name: 70B_prediction, dtype: object

Sample of 70B_prediction_extracted (extracted):
0    D
1    G
2    B
3    D
4    H
5    B
6    C
7    B
8    C
9    D
Name: 70B_prediction_extracted, dtype: object

Null counts:
  Raw column: 3 / 1300
  Extracted column: 4 / 1300

Unique values in 70B_prediction_extracted:
70B_prediction_extracted
C       331
B       294
D       269
A       216
F        38
J        34
H        33
I        30
E        29
G        22
None      4
Name: count, dtype: int64

First 3 non-null raw values from 70B_prediction:
  [0] '<answer>Opti

In [40]:
import pandas as pd
import re

# Load the model predictions
original_df = pd.read_csv(paths.TABLES / "gpt5_spurious_factors.csv")

# Extract the predicted letter from the XML format
def extract_letter_from_xml(pred):
    if isinstance(pred, str):
        match = re.search(r"<answer>Option\s+\[?([A-J])\]?</answer>", pred, re.IGNORECASE)
        if match:
            return match.group(1).upper()
    return None

# Apply extraction to columns that need it
columns_to_extract = ['GPT5_Removal', 'Qwen14B_Removed', '70B_prediction', 
                      '72B_prediction', 'gpt4o_Removed', 'Trainee Removal']

print("Extracting predictions from XML format...")
for col in columns_to_extract:
    print(f"  Processing {col}...")
    original_df[f'{col}_extracted'] = original_df[col].apply(extract_letter_from_xml)

# Show summary of extraction
print("\n" + "="*80)
print("EXTRACTION SUMMARY")
print("="*80)
for col in columns_to_extract:
    original_count = original_df[col].notna().sum()
    extracted_count = original_df[f'{col}_extracted'].notna().sum()
    null_after_extraction = original_count - extracted_count
    
    print(f"\n{col}:")
    print(f"  Original non-null: {original_count}")
    print(f"  Extracted non-null: {extracted_count}")
    print(f"  Failed extractions: {null_after_extraction}")

# Save to CSV
output_filename = paths.TABLES / "gpt5_spurious_factors_with_extractions.csv"
original_df.to_csv(output_filename, index=False)

print("\n" + "="*80)
print(f"SUCCESS: Saved to '{output_filename}'")
print("="*80)
print(f"\nNew columns added:")
for col in columns_to_extract:
    print(f"  - {col}_extracted")

print(f"\nTotal columns in file: {len(original_df.columns)}")
print(f"Total rows: {len(original_df)}")

# Show first few rows of the extracted columns
print("\n" + "="*80)
print("PREVIEW OF EXTRACTED COLUMNS")
print("="*80)
preview_cols = ['Original', 'answer_corr'] + [f'{col}_extracted' for col in columns_to_extract]
print(original_df[preview_cols].head(10))

Extracting predictions from XML format...
  Processing GPT5_Removal...
  Processing Qwen14B_Removed...
  Processing 70B_prediction...
  Processing 72B_prediction...
  Processing gpt4o_Removed...
  Processing Trainee Removal...

EXTRACTION SUMMARY

GPT5_Removal:
  Original non-null: 1300
  Extracted non-null: 1300
  Failed extractions: 0

Qwen14B_Removed:
  Original non-null: 1300
  Extracted non-null: 1300
  Failed extractions: 0

70B_prediction:
  Original non-null: 1297
  Extracted non-null: 1296
  Failed extractions: 1

72B_prediction:
  Original non-null: 1297
  Extracted non-null: 1297
  Failed extractions: 0

gpt4o_Removed:
  Original non-null: 1300
  Extracted non-null: 1300
  Failed extractions: 0

Trainee Removal:
  Original non-null: 1300
  Extracted non-null: 1300
  Failed extractions: 0

SUCCESS: Saved to 'gpt5_spurious_factors_with_extractions.csv'

New columns added:
  - GPT5_Removal_extracted
  - Qwen14B_Removed_extracted
  - 70B_prediction_extracted
  - 72B_prediction_e

In [51]:
import pandas as pd

# Load both dataframes
llama70B_df = pd.read_csv(paths.PREDICTIONS / "gpt5_predictions_on_Llama70B_removed.csv")

print(f"original_df length before merge: {len(original_df)}")
print(f"llama70B_df length: {len(llama70B_df)}")
print()

# Rename the column during merge for clarity
llama70B_subset = llama70B_df[["Origin", "gpt5_direct_prediction"]].copy()
llama70B_subset.rename(columns={"gpt5_direct_prediction": "llama70B_Answer"}, inplace=True)

# Merge the dataframes on "Origin" column
# Using left merge to keep all rows from original_df
original_df = original_df.merge(llama70B_subset, on="Origin", how="left")

print(f"original_df length after merge: {len(original_df)}")
print()

# Check for any rows where llama70B_Answer is NaN (missing matches)
missing_matches = original_df["llama70B_Answer"].isna().sum()
print(f"Number of rows with missing llama70B_Answer: {missing_matches}")

if missing_matches > 0:
    print("\nOrigin IDs with missing llama70B_Answer:")
    missing_origins = original_df[original_df["llama70B_Answer"].isna()]["Origin"]
    print(missing_origins.tolist())
    print()

# Display a sample of the merged data
print("Sample of merged data:")
print("=" * 80)
print(original_df[["Origin", "llama70B_Answer"]].head(10))
print("=" * 80)
print()

# Save the merged dataframe to a new CSV file
output_filename = paths.TABLES / "Llama70B_predictions_on_MedGemma_with_llama70B_Answer.csv"
original_df.to_csv(output_filename, index=False)
print(f"Merged data saved to '{output_filename}'")

# Display column names to verify
print(f"\nColumns in merged dataframe: {original_df.columns.tolist()}")

original_df length before merge: 1300
llama70B_df length: 1297

original_df length after merge: 1300

Number of rows with missing llama70B_Answer: 3

Origin IDs with missing llama70B_Answer:
['ID0291', 'ID0698', 'ID1617']

Sample of merged data:
   Origin              llama70B_Answer
0  ID0002    <answer>Option D</answer>
1  ID0003  <answer>Option [G]</answer>
2  ID0007    <answer>Option B</answer>
3  ID0009    <answer>Option D</answer>
4  ID0010    <answer>Option H</answer>
5  ID0013    <answer>Option B</answer>
6  ID0015    <answer>Option C</answer>
7  ID0016    <answer>Option B</answer>
8  ID0017    <answer>Option C</answer>
9  ID0018    <answer>Option D</answer>

Merged data saved to 'Llama70B_predictions_on_MedGemma_with_llama70B_Answer.csv'

Columns in merged dataframe: ['Origin', 'data_source_df3', 'Patient_Profile', 'Low+Irr', 'High', 'question_options_x', 'answer_corr', 'step1_excerpts', 'question_options_y', 'non_none_sentence_count', 'MedGemma27B_answer', 'GPT5_Removal', 'Or

70B MODEL - OVERALL SPURIOUS RATE
Model                     Spurious Rate (%)    Count           Total Correct in Original
--------------------------------------------------------------------------------
70B_prediction                         70.66%            790 / 1118


70B MODEL - SPURIOUS RATES BY DATA SOURCE
Data Source          Spurious Rate (%)    Count           Total Correct in Original
--------------------------------------------------------------------------------
jama                              67.70%            348 / 514
medbullets                        68.56%            133 / 194
medxpert                          77.73%            171 / 220
mmlu                              72.63%            138 / 190


SUMMARY TABLE - 70B MODEL
         Model Data Source  Spurious Rate (%)  Spurious Count  Total Original Correct
70B_prediction     Overall          70.661896             790                    1118
70B_prediction        jama          67.704280             348          

In [49]:
import pandas as pd

# Load both dataframes
llama70B_df = pd.read_csv(paths.PREDICTIONS / "gpt5_predictions_on_Llama70B_removed.csv")
# original_df = pd.read_csv("Llama70B_predictions_on_MedGemma.csv")  # Adjust filename as needed

print(f"llama70B_df length: {len(llama70B_df)}")
print(f"original_df length: {len(original_df)}")
print()

# Get unique Origin values from both dataframes
llama_origins = set(llama70B_df["Origin"].unique())
original_origins = set(original_df["Origin"].unique())

# Find Origin IDs in original_df but not in llama70B_df
missing_in_llama = original_origins - llama_origins

# Find Origin IDs in llama70B_df but not in original_df (just for completeness)
missing_in_original = llama_origins - original_origins

print(f"Number of Origin IDs in original_df but NOT in llama70B_df: {len(missing_in_llama)}")
print(f"Number of Origin IDs in llama70B_df but NOT in original_df: {len(missing_in_original)}")
print()

if len(missing_in_llama) > 0:
    print("Origin IDs in original_df but NOT in llama70B_df:")
    print("=" * 80)
    missing_sorted = sorted(list(missing_in_llama))
    for origin_id in missing_sorted:
        print(origin_id)
    print("=" * 80)
    print()
    
    # Get the full rows from original_df for these missing IDs
    missing_rows = original_df[original_df["Origin"].isin(missing_in_llama)]
    print(f"\nFull rows from original_df with missing Origin IDs:")
    print(missing_rows)
    
    # Save to CSV
    missing_rows.to_csv(paths.TABLES / "missing_origin_ids_in_llama.csv", index=False)
    print(f"\nMissing rows saved to 'missing_origin_ids_in_llama.csv'")
else:
    print("No missing Origin IDs found!")

if len(missing_in_original) > 0:
    print("\n" + "=" * 80)
    print("Origin IDs in llama70B_df but NOT in original_df:")
    print("=" * 80)
    missing_sorted = sorted(list(missing_in_original))
    for origin_id in missing_sorted:
        print(origin_id)
    print("=" * 80)

# Summary statistics
print("\n" + "=" * 80)
print("SUMMARY:")
print("=" * 80)
print(f"Total unique Origin IDs in original_df: {len(original_origins)}")
print(f"Total unique Origin IDs in llama70B_df: {len(llama_origins)}")
print(f"Origin IDs only in original_df: {len(missing_in_llama)}")
print(f"Origin IDs only in llama70B_df: {len(missing_in_original)}")
print(f"Origin IDs in both: {len(original_origins & llama_origins)}")
print("=" * 80)

llama70B_df length: 1297
original_df length: 1300

Number of Origin IDs in original_df but NOT in llama70B_df: 3
Number of Origin IDs in llama70B_df but NOT in original_df: 0

Origin IDs in original_df but NOT in llama70B_df:
ID0291
ID0698
ID1617


Full rows from original_df with missing Origin IDs:
      Origin data_source_df3  \
174   ID0291        medxpert   
434   ID0698        medxpert   
1043  ID1617        medxpert   

                                        Patient_Profile  \
174   1. A 39-year-old male presents to the clinic c...   
434   1. A 56-year-old female presents to the clinic...   
1043  1. A 23-year-old woman presents to the emergen...   

                                                Low+Irr  \
174   ['3. He denies loss of consciousness and revie...   
434   ['3. She denies recent life changes or increas...   
1043  ['2. She denies fever, diarrhea, vaginal bleed...   

                                                   High  \
174   ['1. A 39-year-old male present

In [55]:
import pandas as pd
import re

# Load the model predictions
original_df = pd.read_csv(paths.TABLES / "gpt5_spurious_factors.csv")

# Extract the predicted letter from the XML format - UPDATED to handle brackets
def extract_letter_from_xml(pred):
    if isinstance(pred, str):
        # Try to match with optional brackets around the letter
        match = re.search(r"<answer>Option\s+\[?([A-J])\]?</answer>", pred, re.IGNORECASE)
        if match:
            return match.group(1).upper()
    return None

# Apply extraction to columns that need it (all except MedGemma27B_answer and Original)
columns_to_extract = ['GPT5_Removal', 'Qwen14B_Removed', '70B_prediction',
                      '72B_prediction', 'gpt4o_Removed', 'Trainee Removal']

for col in columns_to_extract:
    original_df[f'{col}_extracted'] = original_df[col].apply(extract_letter_from_xml)

# Define comparison columns
# MedGemma27B_answer and Original don't need extraction, others do
comparison_columns = {
    'MedGemma27B_answer': 'MedGemma27B_answer',  # No extraction needed
    'GPT5_Removal': 'GPT5_Removal_extracted',
    'Qwen14B_Removed': 'Qwen14B_Removed_extracted',
    '70B_prediction': '70B_prediction_extracted',
    '72B_prediction': '72B_prediction_extracted',
    'gpt4o_Removed': 'gpt4o_Removed_extracted',
    'Trainee Removal': 'Trainee Removal_extracted'
}

# Calculate spurious rate for each column
def calculate_spurious_rate(df, original_col, comparison_col, correct_answer_col):
    """
    Calculate the percentage of questions that were:
    1. Answered CORRECTLY in the original_col (Original == answer_corr)
    2. Answered INCORRECTLY in the comparison_col (comparison_col != answer_corr)
    
    Spurious rate = (count of [Original correct AND comparison incorrect]) / (count of [Original correct])
    """
    # Step 1: Filter rows where Original is correct and has valid data
    original_correct_mask = (df[original_col].notna()) & (df[original_col] == df[correct_answer_col])
    original_correct = df[original_correct_mask].copy()
    
    total_original_correct = len(original_correct)
    
    if total_original_correct == 0:
        return 0.0, 0, 0
    
    # Step 2: Among those where Original was correct, find where comparison is incorrect
    # Comparison is incorrect if: (comparison_col is not null) AND (comparison_col != correct_answer_col)
    comparison_incorrect_mask = (
        original_correct[comparison_col].notna() & 
        (original_correct[comparison_col] != original_correct[correct_answer_col])
    )
    
    spurious_count = comparison_incorrect_mask.sum()
    spurious_rate = (spurious_count / total_original_correct) * 100
    
    return spurious_rate, spurious_count, total_original_correct

# Overall spurious rates
print("="*80)
print("OVERALL SPURIOUS RATES")
print("="*80)
print(f"{'Model':<25} {'Spurious Rate (%)':<20} {'Count':<15} {'Total Correct in Original'}")
print("-"*80)

overall_results = {}
for display_name, col_name in comparison_columns.items():
    rate, count, total = calculate_spurious_rate(
        original_df, 
        'Original',
        col_name, 
        'answer_corr'
    )
    overall_results[display_name] = {
        'rate': rate,
        'count': count,
        'total': total
    }
    print(f"{display_name:<25} {rate:>18.2f}% {count:>14} / {total}")

print("\n")

# Breakdown by data source
print("="*80)
print("SPURIOUS RATES BY DATA SOURCE")
print("="*80)

data_sources = original_df['data_source_df3'].dropna().unique()
breakdown_results = {}

for source in sorted(data_sources):
    print(f"\n--- {source} ---")
    print(f"{'Model':<25} {'Spurious Rate (%)':<20} {'Count':<15} {'Total Correct in Original'}")
    print("-"*80)
    
    source_df = original_df[original_df['data_source_df3'] == source]
    breakdown_results[source] = {}
    
    for display_name, col_name in comparison_columns.items():
        rate, count, total = calculate_spurious_rate(
            source_df,
            'Original',
            col_name,
            'answer_corr'
        )
        breakdown_results[source][display_name] = {
            'rate': rate,
            'count': count,
            'total': total
        }
        print(f"{display_name:<25} {rate:>18.2f}% {count:>14} / {total}")

# Create summary DataFrame for easy export
summary_data = []
for model, stats in overall_results.items():
    summary_data.append({
        'Model': model,
        'Data Source': 'Overall',
        'Spurious Rate (%)': stats['rate'],
        'Spurious Count': stats['count'],
        'Total Original Correct': stats['total']
    })

for source, models in breakdown_results.items():
    for model, stats in models.items():
        summary_data.append({
            'Model': model,
            'Data Source': source,
            'Spurious Rate (%)': stats['rate'],
            'Spurious Count': stats['count'],
            'Total Original Correct': stats['total']
        })

summary_df = pd.DataFrame(summary_data)

# Pivot table for easier viewing
pivot_table = summary_df.pivot(
    index='Model',
    columns='Data Source',
    values='Spurious Rate (%)'
)

print("\n")
print("="*80)
print("SUMMARY PIVOT TABLE (Spurious Rate %)")
print("="*80)
print(pivot_table.round(2))

# Additional diagnostic info
print("\n")
print("="*80)
print("DIAGNOSTIC INFO")
print("="*80)
print(f"Total rows in dataset: {len(original_df)}")
print(f"Rows with non-null 'Original': {original_df['Original'].notna().sum()}")
print(f"Rows where 'Original' is correct: {(original_df['Original'] == original_df['answer_corr']).sum()}")

# Save results
summary_df.to_csv(paths.TABLES / 'spurious_rate_analysis.csv', index=False)
pivot_table.to_csv(paths.TABLES / 'spurious_rate_pivot.csv')

print("\n\nResults saved to:")
print("  - spurious_rate_analysis.csv (detailed)")
print("  - spurious_rate_pivot.csv (pivot table)")

# Optional: Create a cleaner table for the paper
print("\n")
print("="*80)
print("LATEX-READY TABLE (Spurious Rate % by Data Source)")
print("="*80)
if all(col in pivot_table.columns for col in ['jama', 'medbullets', 'medxpert', 'mmlu']):
    print(pivot_table[['jama', 'medbullets', 'medxpert', 'mmlu']].round(1).to_latex())

OVERALL SPURIOUS RATES
Model                     Spurious Rate (%)    Count           Total Correct in Original
--------------------------------------------------------------------------------
MedGemma27B_answer                      7.33%             82 / 1118
GPT5_Removal                            6.53%             73 / 1118
Qwen14B_Removed                         5.46%             61 / 1118
70B_prediction                         11.81%            132 / 1118
72B_prediction                         11.00%            123 / 1118
gpt4o_Removed                           5.90%             66 / 1118
Trainee Removal                         5.99%             67 / 1118


SPURIOUS RATES BY DATA SOURCE

--- jama ---
Model                     Spurious Rate (%)    Count           Total Correct in Original
--------------------------------------------------------------------------------
MedGemma27B_answer                      7.20%             37 / 514
GPT5_Removal                            6.61%   

In [ ]:
import sys
from pathlib import Path

_SANKEY = None
for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "Figures" / "sankey" / "load_expert933_sankey_data.py").is_file():
        _SANKEY = _base / "Figures" / "sankey"
        break
if _SANKEY is None:
    raise FileNotFoundError(
        "Locate repo root (folder containing Figures/sankey/) from cwd for Expert-933 Sankey data."
    )
sys.path.insert(0, str(_SANKEY))
from load_expert933_sankey_data import load_sankey_payload

_sankey = load_sankey_payload()
models = _sankey["models"]
datasets = _sankey["datasets"]
overall_max_length = _sankey["overall_max_length"]
# Human baseline track: not produced by compute_sankey_spurious_data.py; adjust if you have label-level flips.
physicians_mock = {"round1_correct": overall_max_length, "round2_incorrect": 72}


In [ ]:
import sys
from pathlib import Path

_SANKEY = None
for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "Figures" / "sankey" / "load_expert933_sankey_data.py").is_file():
        _SANKEY = _base / "Figures" / "sankey"
        break
if _SANKEY is None:
    raise FileNotFoundError(
        "Locate repo root (folder containing Figures/sankey/) from cwd for Expert-933 Sankey data."
    )
sys.path.insert(0, str(_SANKEY))
from load_expert933_sankey_data import load_sankey_payload

_sankey = load_sankey_payload()
models = _sankey["models"]
datasets = _sankey["datasets"]
overall_max_length = _sankey["overall_max_length"]
# Human baseline track: not produced by compute_sankey_spurious_data.py; adjust if you have label-level flips.
physicians_mock = {"round1_correct": overall_max_length, "round2_incorrect": 72}


In [7]:
# %pip install plotly
%pip install kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [kaleido]m8/9 [kaleido]apher]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sys
from pathlib import Path

_SANKEY = None
for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "Figures" / "sankey" / "load_expert933_sankey_data.py").is_file():
        _SANKEY = _base / "Figures" / "sankey"
        break
if _SANKEY is None:
    raise FileNotFoundError(
        "Locate repo root (folder containing Figures/sankey/) from cwd for Expert-933 Sankey data."
    )
sys.path.insert(0, str(_SANKEY))
from load_expert933_sankey_data import load_sankey_payload

_sankey = load_sankey_payload()
models = _sankey["models"]
datasets = _sankey["datasets"]
overall_max_length = _sankey["overall_max_length"]
# Human baseline track: not produced by compute_sankey_spurious_data.py; adjust if you have label-level flips.
physicians_mock = {"round1_correct": overall_max_length, "round2_incorrect": 72}


In [ ]:
import sys
from pathlib import Path

_SANKEY = None
for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "Figures" / "sankey" / "load_expert933_sankey_data.py").is_file():
        _SANKEY = _base / "Figures" / "sankey"
        break
if _SANKEY is None:
    raise FileNotFoundError(
        "Locate repo root (folder containing Figures/sankey/) from cwd for Expert-933 Sankey data."
    )
sys.path.insert(0, str(_SANKEY))
from load_expert933_sankey_data import load_sankey_payload

_sankey = load_sankey_payload()
models = _sankey["models"]
datasets = _sankey["datasets"]
overall_max_length = _sankey["overall_max_length"]
# Human baseline track: not produced by compute_sankey_spurious_data.py; adjust if you have label-level flips.
physicians_mock = {"round1_correct": overall_max_length, "round2_incorrect": 72}


In [ ]:
import sys
from pathlib import Path

_SANKEY = None
for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "Figures" / "sankey" / "load_expert933_sankey_data.py").is_file():
        _SANKEY = _base / "Figures" / "sankey"
        break
if _SANKEY is None:
    raise FileNotFoundError(
        "Locate repo root (folder containing Figures/sankey/) from cwd for Expert-933 Sankey data."
    )
sys.path.insert(0, str(_SANKEY))
from load_expert933_sankey_data import load_sankey_payload

_sankey = load_sankey_payload()
models = _sankey["models"]
datasets = _sankey["datasets"]
overall_max_length = _sankey["overall_max_length"]
# Human baseline track: not produced by compute_sankey_spurious_data.py; adjust if you have label-level flips.
physicians_mock = {"round1_correct": overall_max_length, "round2_incorrect": 72}


In [8]:
import sys
print(sys.executable)
!{sys.executable} -m pip install -U kaleido
!{sys.executable} -m plotly_get_chrome -y

/home/yuexing/miniconda/bin/python


In [ ]:
import sys
from pathlib import Path

_SANKEY = None
for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "Figures" / "sankey" / "load_expert933_sankey_data.py").is_file():
        _SANKEY = _base / "Figures" / "sankey"
        break
if _SANKEY is None:
    raise FileNotFoundError(
        "Locate repo root (folder containing Figures/sankey/) from cwd for Expert-933 Sankey data."
    )
sys.path.insert(0, str(_SANKEY))
from load_expert933_sankey_data import load_sankey_payload

_sankey = load_sankey_payload()
models = _sankey["models"]
datasets = _sankey["datasets"]
overall_max_length = _sankey["overall_max_length"]
# Human baseline track: not produced by compute_sankey_spurious_data.py; adjust if you have label-level flips.
physicians_mock = {"round1_correct": overall_max_length, "round2_incorrect": 72}
